In [0]:
%pip install openpyxl


In [0]:
import pandas as pd

archivo_excel = "/Volumes/workspace/raw/datos/DatosDeVentaDeTiendaDeTelefonos(10milDatos).xlsx"

excel = pd.ExcelFile(archivo_excel)

print(excel.sheet_names)

In [0]:
df_hoja1 = pd.read_excel(
    archivo_excel,
    sheet_name="Hoja1"
)

display(df_hoja1)

In [0]:
df_hoja2 = pd.read_excel(
    archivo_excel,
    sheet_name="Hoja2"
)

display(df_hoja2)

In [0]:
df_hoja3 = pd.read_excel(
    archivo_excel,
    sheet_name="Hoja3"
)

display(df_hoja3)

In [0]:
df_hoja1[df_hoja1.CódigoProducto=="B00J0O5J4Y"]
df_hoja1[df_hoja1.CódigoProducto=="B016B7INC2"].Unidades.sum()

In [0]:
# Agrupar por producto y sumar las unidades
df_resumen_nuevo = df_hoja1.groupby('CódigoProducto')['Unidades'].sum().reset_index()
df_resumen_nuevo.display()

In [0]:
# Unir ambos DataFrames por el código de producto
df_comparacion = df_resumen_nuevo.merge(df_hoja3, on='CódigoProducto', how='outer')

# Crear una columna que verifique si los totales coinciden
df_comparacion['Coincide'] = df_comparacion['Unidades'] == df_comparacion['Vendidos']
df_comparacion.display()

# Parte 1. Inspección inicial con pandas

Antes del profiling automático, siempre conviene revisar:
- forma del dataset
- tipos de datos
- estadísticas descriptivas
- categorías visibles
- valores raros a primera vista

In [0]:
# ============================================================
# 4. Cargar dataset crudo
# ============================================================
df_hoja1.head()

In [0]:
df_hoja1.info()

In [0]:
df_hoja2.info()

In [0]:
df_hoja3.info()

In [0]:
#df_hoja1.describe(include="number").T
df_hoja1.describe().T

In [0]:

#df_hoja2.describe(include="number").T
df_hoja2.describe().T


In [0]:
#df_hoja3.describe(include="number").T
df_hoja3.describe().T

# Parte 3. Profiling manual con pandas

Aquí construimos un kit simple, útil y entendible

In [0]:
# ============================================================
# 5. Funciones auxiliares
# ============================================================

def percent_missing(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula el porcentaje de valores nulos por columna.

    Qué hace:
    - Revisa cada columna del DataFrame.
    - Cuenta qué proporción de sus valores está vacía.
    - Convierte esa proporción a porcentaje.
    - Ordena el resultado de mayor a menor.

    Para qué sirve:
    - Identificar rápidamente qué columnas tienen más datos faltantes.
    - Priorizar problemas de calidad de datos.
    """
    return (
        df.isna().mean().mul(100).round(2)
        .rename("missing_pct")
        .reset_index()
        .rename(columns={"index": "columna"})
        .sort_values("missing_pct", ascending=False)
    )


def duplicate_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Resume la cantidad de filas duplicadas exactas del DataFrame.

    Qué hace:
    - Cuenta cuántas filas tiene el DataFrame.
    - Calcula cuántas filas están repetidas exactamente.
    - Calcula el porcentaje de duplicados exactos sobre el total.

    Para qué sirve:
    - Medir si el dataset tiene registros repetidos.
    - Detectar posibles problemas de carga, integración o captura.
    """
    rows_total = len(df)
    exact_duplicates = int(df.duplicated().sum())
    return pd.DataFrame({
        "filas_totales": [rows_total],
        "duplicados_exactos": [exact_duplicates],
        "pct_duplicados_exactos": [round(exact_duplicates / rows_total * 100, 2)]
    })


def cardinality_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genera un resumen de cardinalidad por columna.

    Qué hace:
    - Recorre cada columna del DataFrame.
    - Obtiene su tipo de dato.
    - Cuenta cuántos nulos tiene.
    - Cuenta cuántos valores únicos contiene.
    - Calcula qué porcentaje representan esos valores únicos respecto al total de filas.

    Para qué sirve:
    - Identificar columnas casi únicas, como IDs.
    - Detectar columnas categóricas con pocas clases.
    - Encontrar columnas con demasiada variabilidad o posibles inconsistencias.
    """
    rows = []
    for col in df.columns:
        nunique = df[col].nunique(dropna=True)
        rows.append({
            "columna": col,
            "dtype": str(df[col].dtype),
            "nulos": int(df[col].isna().sum()),
            "unicos": int(nunique),
            "pct_unicos_sobre_filas": round(nunique / len(df) * 100, 2)
        })
    return pd.DataFrame(rows).sort_values(
        ["pct_unicos_sobre_filas", "columna"],
        ascending=[False, True]
    )


def iqr_outlier_summary(df: pd.DataFrame, numeric_cols=None) -> pd.DataFrame:
    """
    Detecta outliers en columnas numéricas usando la regla del IQR.

    Qué hace:
    - Si no se indican columnas, toma todas las numéricas.
    - Para cada columna calcula:
        * Q1 (percentil 25)
        * Q3 (percentil 75)
        * IQR = Q3 - Q1
        * límite inferior = Q1 - 1.5 * IQR
        * límite superior = Q3 + 1.5 * IQR
    - Cuenta cuántos valores quedan fuera de esos límites.
    - Calcula el porcentaje de outliers por columna.

    Para qué sirve:
    - Detectar valores extremos o atípicos.
    - Revisar posibles errores de captura o casos inusuales.
    - Priorizar revisión en variables numéricas.
    """
    if numeric_cols is None:
        numeric_cols = df.select_dtypes(include="number").columns.tolist()

    rows = []
    for col in numeric_cols:
        s = df[col].dropna()
        if len(s) == 0:
            continue

        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = ((s < lower) | (s > upper)).sum()

        rows.append({
            "columna": col,
            "q1": round(float(q1), 3),
            "q3": round(float(q3), 3),
            "iqr": round(float(iqr), 3),
            "limite_inferior": round(float(lower), 3),
            "limite_superior": round(float(upper), 3),
            "outliers_iqr": int(outliers),
            "pct_outliers_iqr": round(outliers / len(s) * 100, 2)
        })

    return pd.DataFrame(rows).sort_values("pct_outliers_iqr", ascending=False)


def top_values(df: pd.DataFrame, col: str, n: int = 10) -> pd.DataFrame:
    """
    Muestra los valores más frecuentes de una columna.

    Qué hace:
    - Cuenta cuántas veces aparece cada valor en la columna.
    - Incluye también los nulos.
    - Devuelve los n valores más frecuentes.

    Para qué sirve:
    - Revisar rápidamente la distribución de categorías.
    - Detectar valores dominantes, categorías raras o inconsistentes.
    - Entender mejor columnas de texto o categóricas.
    """
    return (
        df[col].value_counts(dropna=False).head(n)
        .rename("conteo").reset_index()
        .rename(columns={"index": col})
    )


def valid_email_simple(s: pd.Series) -> pd.Series:
    """
    Valida de forma básica si una serie tiene correos con estructura válida.

    Qué hace:
    - Reemplaza nulos por cadena vacía.
    - Evalúa cada valor con una expresión regular simple.
    - Devuelve una serie booleana:
        * True  -> parece un correo válido
        * False -> no cumple la estructura esperada

    Para qué sirve:
    - Detectar correos mal formados de manera rápida.
    - Aplicar una regla básica de calidad sobre columnas de email.

    Nota:
    - Es una validación simple de formato.
    - No garantiza que el correo exista realmente.
    - La expresión regular valida que:
      El texto debe tener algo antes de la arroba, luego una arroba, luego algo antes del punto, luego un punto, luego algo después del punto;
      y no debe tener espacios ni arrobas adicionales en esas partes
    """
    return s.fillna("").str.contains(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", regex=True)

In [0]:

# ============================================================
# 6. Métricas básicas de profiling
# ============================================================
from IPython.display import display, HTML

missing_df = percent_missing(df_hoja1)

dups_df = duplicate_summary(df_hoja1)

card_df = cardinality_summary(df_hoja1)

outliers_df = iqr_outlier_summary(df_hoja1)


display(HTML("<h3>Valores nulos</h3>"))
display(missing_df)
display(HTML("<br>"))

display(HTML("<h3>Duplicados</h3>"))
display(dups_df)
display(HTML("<br>"))

display(HTML("<h3>Cardinalidad</h3>"))
display(card_df.head(15))
display(HTML("<br>"))

display(HTML("<h3>Outliers</h3>"))
display(outliers_df)

In [0]:
categorical_cols = df_hoja1.select_dtypes(include=["object"]).columns.tolist()

print(categorical_cols)

for col in categorical_cols:
    print("\n" + "="*80)
    print(f"Top valores de: {col}")
    display(top_values(df_hoja1, col, n=10))

In [0]:
fecha_min = df_hoja1["Fecha"].min()
fecha_max = df_hoja1["Fecha"].max()

print("Fecha mínima:", fecha_min)
print("Fecha máxima:", fecha_max)

fecha_corte = pd.Timestamp("2015-11-30")
fechas_futuras = (df_hoja1["Fecha"] > fecha_corte).sum()
print("Registros con fecha futura respecto a 2025-03-30:", fechas_futuras)

In [0]:

# ============================================================
# 6. Métricas básicas de profiling
# ============================================================
from IPython.display import display, HTML

missing_df = percent_missing(df_hoja3)

dups_df = duplicate_summary(df_hoja3)

card_df = cardinality_summary(df_hoja3)

outliers_df = iqr_outlier_summary(df_hoja3)


display(HTML("<h3>Valores nulos</h3>"))
display(missing_df)
display(HTML("<br>"))

display(HTML("<h3>Duplicados</h3>"))
display(dups_df)
display(HTML("<br>"))

display(HTML("<h3>Cardinalidad</h3>"))
display(card_df.head(15))
display(HTML("<br>"))

display(HTML("<h3>Outliers</h3>"))
display(outliers_df)

In [0]:
categorical_cols = df_hoja3.select_dtypes(include=["object"]).columns.tolist()

print(categorical_cols)

for col in categorical_cols:
    print("\n" + "="*80)
    print(f"Top valores de: {col}")
    display(top_values(df_hoja3, col, n=12))

In [0]:
df_hoja3.CódigoProducto.str.len()
df_hoja3

In [0]:
#raw.loc[raw["CódigoProducto"] == "B00K15Q2B0", "Almacen"] = 1000


# Parte 4. Reglas de calidad del negocio

Además del profiling estadístico, conviene definir reglas de negocio.

In [0]:
# ============================================================
# 7. Reglas de calidad
# ============================================================
raw=df_hoja3
reglas = {
   # "edad_valida_18_100": raw["edad_cliente"].between(18, 100, inclusive="both"),
    "precio_venta_no_negativo": raw["Precio de venta"]>=  raw["Costo de venta"],
    "precio_positivo": raw["Precio de venta"] > 0,
    "Almacen_positivo": raw["Almacen"] >= 0,
    "Vendidos_positivo": raw["Vendidos"] >= 0,
    "Almacen_Vendidos": raw["Almacen"]>=  raw["Vendidos"],

    #"descuento_entre_0_y_1": raw["descuento_pct"].between(0, 1, inclusive="both"),
    #"fecha_no_futura": raw["Fecha"] <= pd.Timestamp("2026-08-31"),
    #"email_valido_simple": valid_email_simple(raw["email"])
}

quality_rows = []
for nombre, serie in reglas.items():
    invalidos = int((~serie).sum())
    quality_rows.append({
        "regla": nombre,
        "filas_invalidas": invalidos,
        "pct_invalidas": round(invalidos / len(raw) * 100, 2)
    })

quality_df = pd.DataFrame(quality_rows).sort_values("pct_invalidas", ascending=False)
#quality_df.to_csv(DATA_QUALITY_CSV, index=False)
quality_df

# Parte 6. Documentación del dataset

No basta con perfilar: también hay que dejar el dataset explicado.

In [0]:
from pathlib import Path
import pandas as pd

DOCUMENTATION_PATH = Path("/Volumes/workspace/raw/datos/Documentacion")
NAME_DOC=Path("Diccionario_hoja3_producto.csv")#editar
raw=df_hoja3#editar
descripciones = {
    "CódigoProducto": "Identificador único del producto.",
    "Descripción": "Nombre o descripción comercial del producto.",
    "Precio de venta": "Precio unitario de venta asignado al producto.",
    "Costo de venta": "Costo unitario de adquisición o producción directa del producto.",
    "Almacen": "Cantidad de unidades inicial en el inventario.",
    "Vendidos": "Cantidad de unidades del producto vendidas."
}

In [0]:
NAME_DOC=Path("Diccionario_hoja2_representante.csv")
raw=df_hoja2
descripciones = {
    "Representante": "Nombre del representante.",
    "Ciudad": "Ciudad donde se encuentra el representante.",
    "Fotografía": "URL de la fotografía del representante."
}

In [0]:
NAME_DOC=Path("Diccionario_hoja1_ventas_detalle.csv")
raw=df_hoja1
descripciones = {
    "Fecha": "Fecha de venta.",
    "Representante": "Nombre del representante.",
    "CódigoProducto": "Identificador del producto vendido.",
    "Unidades": "Cantidad de unidades vendidas."
}

In [0]:


# =========================================================
# 1. RUTA DONDE SE GUARDARÁ LA DOCUMENTACIÓN
# =========================================================



DOCUMENTATION_PATH.mkdir(
    parents=True,
    exist_ok=True
)

DATA_DICTIONARY_CSV = (
    DOCUMENTATION_PATH / NAME_DOC
)


# =========================================================
# 3. OBTENER EJEMPLOS DE CADA COLUMNA
# =========================================================
ejemplos = {}
for col in raw.columns:
    valores = raw[col].dropna().astype(str).head(3).tolist()
    ejemplos[col] = " | ".join(valores)

# =========================================================
# 4. CREAR EL DICCIONARIO DE DATOS
# =========================================================

diccionario = pd.DataFrame({
    "columna": raw.columns,
    "descripcion": [descripciones.get(c, "") for c in raw.columns],
    "dtype_pandas": [str(raw[c].dtype) for c in raw.columns],
    "nulos": [int(raw[c].isna().sum()) for c in raw.columns],
    "pct_nulos": [round(raw[c].isna().mean() * 100, 2) for c in raw.columns],
    "unicos": [int(raw[c].nunique(dropna=True)) for c in raw.columns],
    "ejemplos": [ejemplos[c] for c in raw.columns]
})


# =========================================================
# 5. GUARDAR COMO CSV
# =========================================================

diccionario.to_csv(
    DATA_DICTIONARY_CSV,
    index=False,
    encoding="utf-8-sig"
)

# =========================================================
# 6. VALIDACIÓN
# =========================================================

print("Diccionario guardado en:")
print(DATA_DICTIONARY_CSV)

print("\n¿El archivo existe?")
print(DATA_DICTIONARY_CSV.exists())

display(diccionario)

In [0]:
%sql
select *from workspace.bronze.tbl_ventas_detalle a
 left join  workspace.bronze.tbl_productos b 
    on(upper(trim(b.`CódigoProducto`))=upper(trim(a.`CódigoProducto`)))
where
    b.`CódigoProducto` is null

;